In [0]:
%run /Users/scottsbv@gmail.com/student-engagement/utils

In [0]:
# Create leads_processed_custom from bronze leads_raw
# Calls extract_custom with dry_run=False to actually WRITE the child table
import time
start = time.time()

print(f"{'='*80}")
print(f"📥 leads_raw → leads_processed_custom")
print(f"{'='*80}")

# Check CDC watermark for the custom table
watermark = get_silver_watermark('leads_processed_custom')
if watermark:
    print(f"  CDC MODE: reading changes after {watermark}")
else:
    print(f"  FULL LOAD: first run")

# Step 1: Parse bronze table
df_flat = parse_bronze_table('leads_raw', watermark=watermark)
if df_flat is None:
    print(f"  ✅ CDC: No new rows since {watermark}")
    print(f"{'='*80}")
else:
    # Step 2: Flatten structs (keep custom struct as-is for extraction)
    df_flat = flatten_structs(df_flat, skip_custom=True)

    # Step 3: Find PK and build carry columns
    pk = find_pk(df_flat)
    carry = list(dict.fromkeys(["bronze_insert_date", "bronze_updated_at"] + ([pk] if pk else [])))

    # Step 4: Extract custom with dry_run=False to actually write the table
    has_custom = "custom" in df_flat.columns and isinstance(df_flat.schema["custom"].dataType, StructType)
    has_custom_cf = any(c.startswith("custom_cf_") for c in df_flat.columns)

    if has_custom or has_custom_cf:
        df_flat = extract_custom(df_flat, 'leads_processed', carry, "1_1", parent_pk=pk, dry_run=False)
        print(f"\n  📋 LINEAGE TREE:")
        for num, depth, name in _lineage_tree:
            print(f"  # * {'---' * depth}{num}_{name}")
    else:
        print("  ⚠️ No custom fields found")

    # Clean up temp tables
    for temp in _TEMP_TABLES:
        spark.sql(f"DROP TABLE IF EXISTS {temp}")
    _TEMP_TABLES.clear()

    print(f"\n  ⏱️ Completed in {time.time() - start:.1f}s")
    print(f"{'='*80}")